<a href="https://colab.research.google.com/github/meghanavanamala/predictiveanalysis/blob/customer_costprediction_seasonally/customer_costprediction_seasonally.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# Simulate data
np.random.seed(42)
n_samples = 500

# Simulated features
df = pd.DataFrame({
    'Product': np.random.choice(['Milk', 'Bread', 'Eggs', 'Biscuits', 'Soap', 'Juice'], n_samples),
    'Season': np.random.choice(['Summer', 'Winter', 'Monsoon'], n_samples),
    'Competitor_Price': np.round(np.random.uniform(10, 100, n_samples), 2),
    'Customer_Sensitivity': np.round(np.random.uniform(0.5, 1.5, n_samples), 2),  # 1 = normal
    'Demand_Index': np.round(np.random.uniform(50, 200, n_samples), 2)  # Proxy demand
})

# Define base and seasonal price effects
season_factor = {'Summer': 0.95, 'Winter': 1.05, 'Monsoon': 1.0}
base_price = {'Milk': 40, 'Bread': 30, 'Eggs': 50, 'Biscuits': 60, 'Soap': 35, 'Juice': 70}

df['Season_Factor'] = df['Season'].map(season_factor)
df['Base_Price'] = df['Product'].map(base_price)

# Optimal Price (target): simulated logic
df['Optimal_Price'] = (
    df['Base_Price'] * df['Season_Factor'] +
    0.3 * df['Competitor_Price'] -
    0.1 * df['Demand_Index'] * df['Customer_Sensitivity'] / 100
)

# Prepare features
df_encoded = pd.get_dummies(df[['Product', 'Season']], drop_first=True)
features = pd.concat([df_encoded, df[['Competitor_Price', 'Customer_Sensitivity', 'Demand_Index']]], axis=1)
target = df['Optimal_Price']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.2, random_state=42)

# Train Random Forest
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Predict and evaluate
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
print(f"\n📉 Mean Squared Error: {mse:.2f}")

# Show sample predictions
sample = pd.DataFrame({
    'Product': df.loc[y_test.index, 'Product'].values[:10],
    'Season': df.loc[y_test.index, 'Season'].values[:10],
    'Predicted Price': np.round(y_pred[:10], 2),
    'Actual Price': np.round(y_test.values[:10], 2)
})

print("\n🔍 Sample Predictions:")
print(sample)



📉 Mean Squared Error: 1.99

🔍 Sample Predictions:
    Product   Season  Predicted Price  Actual Price
0      Soap   Winter            61.98         62.90
1  Biscuits   Summer            72.53         71.70
2     Bread  Monsoon            40.71         41.42
3      Soap  Monsoon            60.87         59.62
4      Milk  Monsoon            65.84         65.09
5     Bread   Summer            49.40         48.26
6     Juice   Summer            82.55         77.97
7      Eggs  Monsoon            78.37         78.50
8     Bread   Summer            46.43         45.56
9      Eggs   Summer            67.83         66.08
